In [1]:
import os
import glob
import json
import cv2
import torch
import numpy as np
from PIL import Image
from pathlib import Path
from transformers import OwlViTProcessor, OwlViTForObjectDetection

ROOT_DIR = Path("../").resolve()
DATA_DIR = ROOT_DIR / "data"
WEIGHTS_DIR = ROOT_DIR / "weights"

/home/octoopt/anaconda3/envs/turlio/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-11-16 17:19:41.563974: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
def read_json(filepath):
    """
    Read and parse a JSON file.

    Args:
        filepath (str): Path to the JSON file

    Returns:
        dict/list: Parsed JSON data

    Raises:
        FileNotFoundError: If file doesn't exist
        json.JSONDecodeError: If file contains invalid JSON
    """
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)
        return data
    except FileNotFoundError:
        print(f"Error: File '{filepath}' not found")
        raise
    except json.JSONDecodeError as e:
        print(f"Error: Invalid JSON in '{filepath}': {e}")
        raise
    except Exception as e:
        print(f"Error reading file: {e}")
        raise


def write_json(filepath, data, indent=2, ensure_dir=True):
    """
    Write data to a JSON file.

    Args:
        filepath (str): Path to save the JSON file
        data (dict/list): Data to write
        indent (int): Indentation level for pretty printing (default: 2)
        ensure_dir (bool): Create directory if it doesn't exist (default: True)

    Returns:
        bool: True if successful, False otherwise
    """
    try:
        # Create directory if needed
        if ensure_dir:
            directory = os.path.dirname(filepath)
            if directory and not os.path.exists(directory):
                os.makedirs(directory)

        # Write JSON file
        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=indent, ensure_ascii=False)

        return True
    except Exception as e:
        print(f"Error writing to '{filepath}': {e}")
        return False

In [3]:
import os
import json
import cv2
import torch
import numpy as np
from PIL import Image
from pathlib import Path
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

# ==============================================================
# Simple IoU-based tracker
# ==============================================================


class SimpleTracker:
    def __init__(self, iou_threshold=0.4):
        self.next_id = 1
        self.tracks = {}  # track_id -> last_bbox
        self.iou_threshold = iou_threshold

    @staticmethod
    def iou(boxA, boxB):
        x1 = max(boxA[0], boxB[0])
        y1 = max(boxA[1], boxB[1])
        x2 = min(boxA[2], x2 := boxB[2])
        y2 = min(boxA[3], y2 := boxB[3])

        inter = max(0, x2 - x1) * max(0, y2 - y1)
        if inter <= 0:
            return 0.0

        areaA = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
        areaB = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
        return inter / (areaA + areaB - inter)

    def update(self, detections):
        """
        detections: list of (bbox, score)
        bbox = [x1, y1, x2, y2]
        """
        new_tracks = {}

        for det_bbox, det_score in detections:
            best_id = None
            best_iou = 0

            for tid, last_bbox in self.tracks.items():
                iou_val = self.iou(det_bbox, last_bbox)
                if iou_val > best_iou:
                    best_iou = iou_val
                    best_id = tid

            if best_iou >= self.iou_threshold:
                new_tracks[best_id] = det_bbox
            else:
                new_tracks[self.next_id] = det_bbox
                self.next_id += 1

        self.tracks = new_tracks
        return self.tracks


# ==============================================================
# LLMDet Only Image-Guided Detector + Tracker
# ==============================================================


class LLMDetTracker:
    def __init__(self, device="cuda"):
        self.device = device

        print("[INIT] Loading LLMDet...")
        model_name = "iSEE-Laboratory/llmdet_large"

        self.processor = AutoProcessor.from_pretrained(model_name)
        self.model = AutoModelForZeroShotObjectDetection.from_pretrained(model_name).to(
            device
        )

        self.ref_images = []
        self.tracker = SimpleTracker()

        print("[READY] LLMDet Tracker loaded.")

    # -------------------------------------------------------------
    # Add reference images
    # -------------------------------------------------------------
    def add_reference_image(self, image_path):
        img = Image.open(image_path).convert("RGB")
        self.ref_images.append(img)
        print(f"[INFO] Added reference image: {image_path}")

    def clear_reference_images(self):
        self.ref_images.clear()
        print("[INFO] Cleared all reference images.")

    # -------------------------------------------------------------
    # Detection with reference images (image-guided)
    # -------------------------------------------------------------
    def detect_similar(self, frame_pil, score_threshold=0.20):
        if len(self.ref_images) == 0:
            return []

        # ⭐ LLMDet-specific: prompt_images
        inputs = self.processor(
            images=frame_pil, prompt_images=self.ref_images, return_tensors="pt"
        ).to(self.device)

        with torch.no_grad():
            outputs = self.model(**inputs)

        h, w = frame_pil.size[1], frame_pil.size[0]
        target_sizes = [[h, w]]

        results = self.processor.post_process(outputs, target_sizes=target_sizes)[0]

        boxes = results["boxes"].cpu().numpy()
        scores = results["scores"].cpu().numpy()

        detections = []
        for box, score in zip(boxes, scores):
            if score >= score_threshold:
                detections.append((box.tolist(), float(score)))

        return detections

    # -------------------------------------------------------------
    # Main tracking loop
    # -------------------------------------------------------------
    def track_video(
        self,
        video_path,
        video_id="result",
        score_threshold=0.20,
        output_dir="results",
        debug=True,
    ):
        os.makedirs(output_dir, exist_ok=True)
        output_path = os.path.join(output_dir, f"{video_id}.json")

        cap = cv2.VideoCapture(video_path)
        assert cap.isOpened(), f"Cannot open: {video_path}"

        collected = []
        frame_idx = 0

        while True:
            ret, frame_bgr = cap.read()
            if not ret:
                break

            frame_idx += 1
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            frame_pil = Image.fromarray(frame_rgb)

            # -----------------------------------------------------
            # LLMDet image-guided detection
            # -----------------------------------------------------
            detections = self.detect_similar(frame_pil, score_threshold)

            # -----------------------------------------------------
            # Track IDs
            # -----------------------------------------------------
            tracks = self.tracker.update(detections)

            # Draw and save
            for tid, bbox in tracks.items():
                x1, y1, x2, y2 = map(int, bbox)

                collected.append(
                    {
                        "frame": frame_idx,
                        "track_id": tid,
                        "x1": x1,
                        "y1": y1,
                        "x2": x2,
                        "y2": y2,
                    }
                )

                if debug:
                    cv2.rectangle(frame_bgr, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    cv2.putText(
                        frame_bgr,
                        f"ID {tid}",
                        (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.7,
                        (0, 255, 0),
                        2,
                    )

            if debug:
                cv2.imshow("LLMDet Tracking", frame_bgr)
                if cv2.waitKey(1) & 0xFF == 27:
                    break

        cap.release()
        if debug:
            cv2.destroyAllWindows()

        # -----------------------------------------------------
        # Save JSON
        # -----------------------------------------------------
        out_data = {"video_id": video_id, "detections": collected}

        with open(output_path, "w") as f:
            json.dump(out_data, f, indent=2)

        print(f"[DONE] Saved: {output_path}")
        return out_data

In [4]:
tracker = LLMDetTracker(device="cuda")

[INIT] Loading LLMDet...


ValueError: The checkpoint you are trying to load has model type `mm-grounding-dino` but Transformers does not recognize this architecture. This could be because of an issue with the checkpoint, or because your version of Transformers is out of date.

You can update Transformers with the command `pip install --upgrade transformers`. If this does not work, and the checkpoint is very new, then there may not be a release version that supports this model yet. In this case, you can get the most up-to-date code by installing Transformers from source with the command `pip install git+https://github.com/huggingface/transformers.git`

In [11]:
for sample in os.listdir(str(DATA_DIR / "public_test" / "samples")):
    VIDEO_ID = sample
    SAMPLE_PATH = DATA_DIR / "public_test" / "samples" / VIDEO_ID
    IMAGE_PATH = SAMPLE_PATH / "object_images"

    images = glob.glob(str(IMAGE_PATH / "*.jpg"))
    video_path = SAMPLE_PATH / "drone_video.mp4"

    print(f"Checking {sample} ...")
    tracker.delete_reference_images()

    # Step 1: Add reference images
    for img in images:
        tracker.add_reference_image(img)

    # Step 2: Track objects in video
    tracker.track_video(
        video_path,
        score_threshold=0.20,
        video_id=VIDEO_ID,
        output_dir="../temp/results/16112025/009",
        debug=False,
    )

Checking BlackBox_1 ...
[INFO] Added reference image: /home/octoopt/workspace/projects/competition/ZAIC2025_AeroEyes/data/public_test/samples/BlackBox_1/object_images/img_2.jpg
[INFO] Added reference image: /home/octoopt/workspace/projects/competition/ZAIC2025_AeroEyes/data/public_test/samples/BlackBox_1/object_images/img_3.jpg
[INFO] Added reference image: /home/octoopt/workspace/projects/competition/ZAIC2025_AeroEyes/data/public_test/samples/BlackBox_1/object_images/img_1.jpg


TypeError: OwlViTForObjectDetection.forward() got an unexpected keyword argument 'query_pixel_values'

In [ ]:
result_files = glob.glob("../temp/results/16112025/009/*.json")
results = []
for file in result_files:
    print(file)
    result = read_json(file)
    results.append(result)


len(results)

In [ ]:
write_json(filepath="../temp/results/16112026_final_results_009.json", data=results)